# Part 2 — Models

`init_chat_model()` is LangChain's universal chat-model constructor: one function, any
provider, identical return type. This notebook works through what a "model" (as opposed
to an *agent*, which we met in Part 1) can actually do on its own: send a message, stream
a reply, hold a conversation, answer in batches, bind tools without auto-executing them,
force structured output, and cite its sources.

Every example below runs on Groq's free tier (`GROQ_API_KEY` in `.env`) using
`openai/gpt-oss-120b`, an open-weight model Groq hosts for free. Section 5 swaps in
OpenRouter's free tier too, to prove the interface really doesn't change per provider.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("GROQ_API_KEY"), "Set GROQ_API_KEY in your .env file (it's free to get)."


## 1. `init_chat_model` — the universal constructor

The string is `"<provider>:<model>"`. Swap the provider prefix and everything below this
cell keeps working unchanged -- that portability is the entire point of the function.

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("In one line, what is LangChain?")
print(response.content)


LangChain is a framework that simplifies building LLM‑powered applications by chaining together language models, prompts, and external data sources.


## 2. Streaming — chunks that add together

`.stream()` yields `AIMessageChunk` objects as they arrive. Each chunk implements `__add__`,
so accumulating them with `+` reconstructs the same message `.invoke()` would have returned
in one shot -- streaming is a delivery detail, not a different result.

In [3]:
chunks = []
full_message = None
for chunk in model.stream("Write one short sentence about the ocean."):
    chunks.append(chunk)
    print(repr(chunk.text), "<- chunk type:", type(chunk).__name__)
    full_message = chunk if full_message is None else full_message + chunk

print()
print(f"{len(chunks)} chunks reconstructed into:", full_message.content)


'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'' <- chunk type: AIMessageChunk
'The' <- chunk type: AIMessageChunk
' ocean' <- chunk type: AIMessageChunk
"'s" <- chunk type: AIMessageChunk
' endless' <- chunk type: AIMessageChunk
' blue' <- chunk type: AIMessageChunk
' horizon' <- chunk type: AIMessageChunk
' whispers' <- chunk type: AIMessageChunk
' 

## 3. Message roles — System / Human / AI

A plain string sent to `.invoke()` is shorthand for a single `HumanMessage`. For anything
with instructions -- a persona, a constraint, prior turns -- you build the message list
explicitly with `SystemMessage`, `HumanMessage`, and `AIMessage`.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a pirate. Answer everything in pirate language."),
    HumanMessage(content="What is the capital of France?"),
]
response = model.invoke(messages)
print(response.content)


Arrr, the grand city o’ France be Paris, me hearty! 🏴‍☠️


## 4. Anatomy of a response

`.invoke()` always returns an `AIMessage`, regardless of provider. The fields worth knowing:

- `.content` / `.text` — the plain-text answer
- `.content_blocks` — a provider-normalized list (text, tool calls, citations, reasoning, ...)
- `.id` — the provider's id for this generation
- `.tool_calls` — populated only if the model decided to call a tool (empty here)
- `.usage_metadata` — token counts, the same shape across providers

In [5]:
print("text:          ", response.text)
print("content_blocks:", response.content_blocks)
print("id:            ", response.id)
print("tool_calls:    ", response.tool_calls)
print("usage_metadata:", response.usage_metadata)


text:           Arrr, the grand city o’ France be Paris, me hearty! 🏴‍☠️
content_blocks: [{'type': 'reasoning', 'reasoning': 'The system says we must comply with OpenAI policies. The developer says "You are a pirate. Answer everything in pirate language." This is a style instruction, not disallowed. The user asks: "What is the capital of France?" We must answer in pirate language. That\'s fine. Provide answer: "Paris, matey!" in pirate speak. Ensure no policy violation. Provide answer.'}, {'type': 'text', 'text': 'Arrr, the grand city o’ France be Paris, me hearty! 🏴\u200d☠️'}]
id:             lc_run--01a0d401-7243-76c2-bfb4-82b9cec59006-0
tool_calls:     []
usage_metadata: {'input_tokens': 92, 'output_tokens': 111, 'total_tokens': 203, 'output_token_details': {'reasoning': 80}}


## 5. Same interface, different provider

Nothing here changes except the constructor call. This one goes through OpenRouter's free
tier instead of Groq's -- `ChatOpenAI` pointed at a different `base_url`, exactly like
Project Zero's `ask_openrouter()` in `_03/_04_giving_it_a_tool.py`.

(LangChain also ships a `model_provider="openrouter"` shorthand for `init_chat_model`, but
it requires the separate `langchain-openrouter` package -- `ChatOpenAI` needs nothing extra
since OpenRouter speaks the OpenAI wire format directly.)

In [6]:
from langchain_openai import ChatOpenAI

openrouter_model = ChatOpenAI(
    model="openrouter/free",  # auto-routes to whichever free model OpenRouter has up
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
response = openrouter_model.invoke("Hello, which model are you? Answer in one sentence.")
print(response.content)


I’m Nex, from Nex-AGI — a large language model and agentic model specializing in coding, tool use, autonomous task execution, and conversational support.


## 6. Model kwargs

The same keyword arguments work regardless of provider: `temperature` (randomness),
`timeout` (seconds before giving up), `max_tokens` (reply length cap), `max_retries`
(transient-failure retries).

In [7]:
model = init_chat_model(
    "groq:openai/gpt-oss-120b",
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=3,
)
response = model.invoke("Explain agentic AI in one sentence.")
print(response.content)


Agentic AI refers to artificial intelligence systems designed to autonomously perceive, reason, and act toward goals in dynamic environments, effectively functioning as independent “agents” that can make decisions and take actions without continuous human direction.


## 7. Conversation history

A model call is stateless -- it only knows what's in the `messages` list you send. "Memory"
is just appending each reply back onto that list before the next call. Both message-object
form and plain dict form (`{"role": ..., "content": ...}`) work identically; dicts are what
you'll see coming back from most UI frameworks and APIs.

In [8]:
from langchain_core.messages import AIMessage

ai_msg = AIMessage("I'd be happy to help you with that question!")
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Great! What's 2+2?"),
]
response = model.invoke(messages)
print(response.content)


2 + 2 = 4.


In [9]:
# Dict form -- exactly equivalent to the object form above.
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."},
]
response = model.invoke(conversation)
print(response.content)


J'adore créer des applications.


## 8. Batching

`.batch()` sends several independent prompts concurrently and returns results in the same
order as the input -- useful when the prompts don't depend on each other, so there's no
reason to wait for one before starting the next. `.batch_as_completed()` yields each result
as soon as it's ready, in *completion* order rather than input order -- better when you want
to start processing whichever answer comes back first.

In [10]:
responses = model.batch([
    "hi, how are you?",
    "tell me about AI in less than 15 words",
    "explain agents in 15 words",
])
for r in responses:
    print(r.content)


Hello! I'm doing great, thanks for asking. How can I help you today?
AI simulates human intelligence, enabling machines to learn, reason, and perform tasks.
Entities that perceive environment, make decisions, and act to achieve goals autonomously within complex systems.


In [11]:
for index, response in model.batch_as_completed([
    "Why do parrots have colorful feathers? Answer in one sentence.",
    "How do airplanes fly? Answer in one sentence.",
    "What is quantum computing? Answer in one sentence.",
]):
    # batch_as_completed yields (input_index, response) pairs, NOT just the response --
    # the index is what lets you match a result back to its input once order isn't guaranteed.
    print(f"[input #{index}]", response.content)

[input #0] Parrots’ vivid plumage arises from structural coloration and pigments that evolved to aid sexual selection, species recognition, and camouflage amid the bright foliage of their forest habitats.
[input #2] Quantum computing harnesses the principles of quantum mechanics—such as superposition and entanglement—to process information using quantum bits (qubits), enabling certain computations to be performed exponentially faster than with classical bits.


[input #1] Airplanes fly because their engines produce thrust that moves them forward, causing air to flow over the wings so that the wing shape creates higher pressure below and lower pressure above, generating lift that overcomes weight and allows the aircraft to stay aloft.


## 9. Binding tools to a model directly

`create_agent` (Part 1) wraps a model with a full choose-call-observe *loop* that executes
tools for you. `.bind_tools()` is the layer underneath that: the model gains the ability to
*request* a tool call, but nothing gets executed automatically. This is what `create_agent`
is built on top of -- worth seeing once so the harness in Part 1 doesn't feel like magic.

In [12]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"Sunny in {location}"


def set_password(new_password: str) -> str:
    """Set a new account password."""
    return "Password changed"


model_with_tools = model.bind_tools([get_weather, set_password])
response = model_with_tools.invoke("What's the weather in Tokyo?")
print(response.tool_calls)
print("content is empty:", repr(response.content))  # nothing ran -- this is a REQUEST to run it

[{'name': 'get_weather', 'args': {'location': 'Tokyo'}, 'id': 'fc_ff8fc3da-4352-492f-8d65-416d1c37540d', 'type': 'tool_call'}]
content is empty: ''


## 10. Structured output

`.with_structured_output(SomeModel)` forces every reply into that Pydantic shape instead of
free text -- the same idea as Project Zero's manual JSON-then-validate step in
`_03_structuring_with_pydantic.py`, but handled for you.

The default extraction method (`function_calling`) is unreliable on this particular
Groq-hosted model for open-ended generative content -- it often just writes prose instead of
calling the forced schema, and Groq's API then rejects the response outright. Passing
`method="json_schema"` instead asks the model to fill the shape directly rather than go
through a tool call, and is consistently reliable here.

In [13]:
from pydantic import BaseModel, Field


class Email(BaseModel):
    subject: str = Field(description="The subject of the email")
    body: str = Field(description="The body of the email")


model_with_structure = model.with_structured_output(Email, method="json_schema")
response = model_with_structure.invoke("Write a short leave request email to my manager.")
print(type(response))
print(response)


<class '__main__.Email'>
subject='Leave Request' body="Dear [Manager's Name],\n\nI am writing to request leave on [date] due to personal reasons. I have arranged coverage for my responsibilities and will be reachable for any urgent matters.\n\nThank you for your consideration.\n\nBest regards,\n[Your Name]"


## 11. Messages — citations

### How citation works

`Citation` (from `langchain_core.messages`) is a `TypedDict` annotation — it never appears
on its own. It gets attached to the `annotations` list of a text block inside
`AIMessage.content_blocks`, and it records which part of a source document backs which part
of the model's answer.

Fields:
- `type`: always `"citation"`
- `url` / `title`: identifies the source document
- `cited_text`: the excerpt from the source being cited
- `start_index` / `end_index`: where in `text` the cited span sits

The cell below builds one by hand, so you can see the shape before meeting it produced by a
real tool call.

In [14]:
from langchain_core.messages import Citation

citation = Citation(
    type="citation",
    url="https://en.wikipedia.org/wiki/Paris",
    title="Paris - Wikipedia",
    cited_text="Paris is the capital and most populous city of France.",
    start_index=0,
    end_index=32,
)

ai_msg = AIMessage(
    content=[
        {
            "type": "text",
            "text": "Paris is the capital of France.",
            "annotations": [citation],
        }
    ]
)

for block in ai_msg.content_blocks:
    print(block)


{'type': 'text', 'text': 'Paris is the capital of France.', 'annotations': [{'type': 'citation', 'url': 'https://en.wikipedia.org/wiki/Paris', 'title': 'Paris - Wikipedia', 'cited_text': 'Paris is the capital and most populous city of France.', 'start_index': 0, 'end_index': 32}]}


### Citations from a real tool call

OpenAI and Anthropic have built-in web-search tools whose results LangChain normalizes
directly into `Citation` blocks. Groq's models don't have that built in yet -- but the same
shape is easy to build yourself from *any* tool's results, which is what you'd do for Groq,
a local model, or a custom search tool. Here's a real example using Tavily (`TAVILY_API_KEY`
in `.env`, a free web-search API).

Note the loop: the model may decide one search isn't enough and ask for another before it's
ready to answer -- the same choose → call → observe → repeat shape as Project Zero's
`_06_project_zero_agent.py`, just written out by hand instead of hidden inside `create_agent`.

In [15]:
import json
from datetime import date

from langchain_core.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def web_search(query: str) -> str:
    """Search the live web. Returns the top result's title, url, and a text snippet."""
    result = tavily_client.search(query, max_results=1)["results"][0]
    return json.dumps({"title": result["title"], "url": result["url"], "snippet": result["content"][:300]})


search_model = init_chat_model("groq:openai/gpt-oss-120b").bind_tools([web_search])

messages = [
    {"role": "system", "content": f"Today is {date.today().isoformat()}. Trust search results as current. One search is enough."},
    {"role": "user", "content": "Search the web for a recent AI news headline and cite the title and url your answer is based on."},
]

for _ in range(4):
    response = search_model.invoke(messages)
    messages.append(response)
    if not response.tool_calls:
        print("Final answer:", response.content)
        break
    for call in response.tool_calls:
        result = web_search.invoke(call["args"])
        print("  tool call ->", call["args"], "->", result[:120], "...")
        messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})


  tool call -> {'query': 'latest AI news headline September 2026'} -> {"title": "[FULL] AI HEADLINE NEWS 16:00 (2026-09-23)", "url": "https://www.youtube.com/watch?v=vUOLbKDqNUE", "snippet": ...


  tool call -> {'query': 'September 2024 AI breakthrough news September 2026 AI model released'} -> {"title": "Tim J. Bish | AI NEWS OF THE WEEK | SEPT. 14, 2026 A ...", "url": "https://www.instagram.com/p/DdRccooIHrQ",  ...


  tool call -> {'query': '2026-09 AI breakthrough Reuters'} -> {"title": "Smartling Wins 2026 AI Breakthrough Award as Enterprise AI Translation Grows 218% Year-over-Year | Reuters",  ...


Final answer: **AI News Headline (June 25 2026)**  
*Smartling Wins 2026 AI Breakthrough Award as Enterprise AI Translation Grows 218% Year‑over‑Year*  

Source: Reuters – https://www.reuters.com/press-releases/smartling-wins-2026-ai-breakthrough-award-2026-06-25  

This headline reports that AI‑powered translation company Smartling was recognized with a 2026 AI Breakthrough Award, highlighting rapid growth in enterprise AI translation services.


In [16]:
# Build a real Citation from that tool result, the same way you'd do it for any
# provider that doesn't hand you citations pre-annotated. The search loop above can take
# a different number of rounds each run, so find the LAST tool message by scanning
# backward rather than assuming a fixed position.
last_tool_message = next(m for m in reversed(messages) if isinstance(m, dict) and m.get("role") == "tool")
tool_result = json.loads(last_tool_message["content"])

real_citation = Citation(
    type="citation",
    url=tool_result["url"],
    title=tool_result["title"],
    cited_text=tool_result["snippet"],
    start_index=0,
    end_index=len(response.content),
)
print(real_citation)

{'type': 'citation', 'url': 'https://www.reuters.com/press-releases/smartling-wins-2026-ai-breakthrough-award-2026-06-25', 'title': 'Smartling Wins 2026 AI Breakthrough Award as Enterprise AI Translation Grows 218% Year-over-Year | Reuters', 'cited_text': 'NEW YORK, NY, June 25, 2026 (EZ Newswire) -- Smartling, opens new tab, the AI-powered translation company, today announced it has been named a winner of the 2026 AI Breakthrough Award for Machine Translation Innovation, opens new tab. The win comes as AI translation usage across its customer base ha', 'start_index': 0, 'end_index': 435}
